# 01 - Retrieval (مطابق لـ `retrieval.py`)

الـ notebook ده بيذاكر نفس production code الموجود في `rag/retrieval.py`: تحميل الـ documents، تنظيف الكلمات، الـ Chroma semantic search، والـ hybrid ranking. أي تعديل حقيقي يتعمل في `.py`، والـ notebook يعرضه ويجربه.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'rag' else Path.cwd()))
import inspect
from rag import retrieval

print('Production module:', retrieval.__file__)
print('Embedding model:', retrieval.EMBEDDING_MODEL)
print('Embeddings enabled:', retrieval.EMBEDDINGS_ENABLED)
print('Books enabled:', retrieval.LOAD_BOOKS)

## 1. تنظيف السؤال إلى terms
الدالة دي بتشيل الكلمات العامة علشان الـ matching يركز على الكلمات المهمة.

In [ ]:
print(inspect.getsource(retrieval.terms))
print(retrieval.terms('What does the ancient Egyptian ankh mean?'))

## 2. تحميل الـ knowledge base
`load_documents()` تجمع catalog الرموز وJSONL وPDF pages وsource manifest في Documents موحدة.

In [ ]:
print(inspect.getsource(retrieval.load_documents))
docs = retrieval.load_documents()
print('Documents loaded:', len(docs))
print('First document:', docs[0].id, '|', docs[0].source)

## 3. الـ Vector Store والـ retrieval
لو Chroma وembeddings مفعّلين، `LocalVectorStore` يعمل semantic search. بعد كده `retrieve()` يدمج semantic similarity مع keyword overlap وexact Gardiner matches.

In [ ]:
print(inspect.getsource(retrieval.LocalVectorStore))
print(inspect.getsource(retrieval.retrieve))

In [ ]:
question = 'What does S34 mean?'
records = retrieval.retrieve(question, codes=['S34'], top_k=5)
print('Question:', question)
for rank, record in enumerate(records, 1):
    print(f'#{rank} {record["id"]} | score={record["score"]} | similarity={record["similarity"]}')
    print(record['text'][:220])

### مذاكرة النتيجة
`S34` ظهر بقوة لأنه exact Gardiner match، وبعده الـ semantic والـ lexical signals بيحددوا ترتيب باقي النتائج.